# 02 — Feature Engineering
Demonstrates the adstock + saturation transforms and the final modeling feature matrix. Production code: `src/features/`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, plotly.graph_objects as go
from src.data.load_data import load_config
from src.data.preprocess import preprocess
from src.features.build_features import build_feature_matrix
from src.features.adstock import GeometricAdstock
from src.features.saturation import HillSaturation

cfg = load_config()
processed = preprocess(cfg)
bundle = build_feature_matrix(processed, cfg)
bundle['X'].head()

## Visualize adstock decay for a few lambda values

In [ ]:
spend = processed['spend_meta'].values.reshape(-1,1)
fig = go.Figure()
fig.add_trace(go.Scatter(y=spend.flatten(), name='raw spend'))
for lam in [0.1, 0.4, 0.7]:
    ad = GeometricAdstock(lambdas=lam).fit_transform(spend)
    fig.add_trace(go.Scatter(y=ad.flatten(), name=f'adstock lambda={lam}'))
fig.show()

## Visualize Hill saturation shapes

In [ ]:
grid = np.linspace(0, 500000, 200).reshape(-1,1)
fig = go.Figure()
for k, s in [(100000,1.2), (150000,1.6), (150000,2.5)]:
    sat = HillSaturation(k=k, s=s).fit_transform(grid)
    fig.add_trace(go.Scatter(x=grid.flatten(), y=sat.flatten(), name=f'k={k}, s={s}'))
fig.update_layout(xaxis_title='adstocked spend', yaxis_title='saturation output (0-1)')
fig.show()